In [ ]:


        # Move elevator


In [ ]:
class Elevator:

  def __init__(self, current_floor=0):
     self.current_floor = current_floor
     self.direction  = "IDLE"
     self.requests = []

  def request_floor(self, floor, request_type):
     self.requests.append(floor)

     if request_type == "UP":
        self.direction = "UP"

     elif request_type == "DOWN":
        self.direction = "DOWN"

     if self.direction == "UP":
        self.requests.sort()
     else:
        self.requests.sort(reverse=True)

     next_floor = self.requests.pop(0)

     while self.current_floor != next_floor:
        if self.current_floor < next_floor:
           self.current_floor += 1
        else:
           self.current_floor -= 1
        print(f"Elevator at floor {self.current_floor}")

     print(f"Arrived at floor {next_floor}, doors opening")

OPTIMIZED WITH SOLID VERSION


In [ ]:
from enum import Enum
from abc import ABC, abstractmethod

In [ ]:
class Direction(Enum):
    UP = 1
    DOWN = 2
    IDLE = 3






In [ ]:
class Request:
  def __init__(self, floor: int):
    self.floor = floor

  def __repr__(self):
    return f"Request(floor = { self.floor})"



In [ ]:
from abc import ABC, abstractmethod

class Display(ABC):
    @abstractmethod
    def show(self, message:str):
      pass

class ConsoleDisplay(Display):
  def show(self, message:str):
     print(message)
class SilentDisplay(Display):
  def show(self, message:str):
    pass

In [9]:
from abc import ABC, abstractmethod

class SchedulingStrategy(ABC):
    @abstractmethod
    def next_stop(self, current_floor: int, requests: list, direction: 'Direction'):
        pass

class NearestFirstStrategy(SchedulingStrategy):

  def next_stop(self, current_floor, requests, direction):
    if not requests:
      return None
    if direction == Direction.UP:
      ahead = [ r for r in requests if r.floor >= current_floor]
      return min(ahead, key=lambda r:r.floor) if ahead else min(requests, key=lambda r:r.floor)

    elif direction == Direction.DOWN:
      behind = [ r for r in requests if r.floor <= current_floor]
      return max(behind, key= lambda r: r.floor) if behind else max(requests, key = lambda r: r.floor)

    else:
       return min(requests, key=lambda r: abs(r.floor - current_floor))
class ScanStrategy(SchedulingStrategy):
   def next_stop(self, current_floor, requests, direction):
       if not requests:
          return None
       if direction == Direction.UP:
          ahead = [r for r in requests if r.floor >= current_floor]
          return min(ahead, key=lambda r: r.floor) if ahead else min(requests, key=lambda r: r.floor)
       elif direction == Direction.DOWN:
          behind = [r for r in requests if r.floor <= current_floor]
          return max(behind, key=lambda r: r.floor) if behind else max(requests, key=lambda r:r.floor)
       else:
          return min(requests, key=lambda r: abs(r.floor- current_floor))

In [12]:
class Elevator:
   def __init__(self, elevator_id: str, strategy: SchedulingStrategy, display: 'Display', current_floor: int = 0):
       self.elevator_id = elevator_id
       self.current_floor = current_floor
       self.direction = Direction.IDLE
       self.requests: list['Request'] = []
       self.strategy = strategy
       self.display = display
   def add_request(self, request: 'Request'):
       self.requests.append(request)
       self.display.show(f"[{self.elevator_id}] Request added for floor {request.floor}")

   def step(self):
       if not self.requests:
           self.direction = Direction.IDLE
           return
       target = self.strategy.next_stop(self.current_floor, self.requests, self.direction)
       if target is None:
           return
       self.direction = Direction.UP if target.floor > self.current_floor else (
           Direction.DOWN if target.floor < self.current_floor else self.direction
       )


       while self.current_floor != target.floor:

            self.current_floor += 1 if self.direction == Direction.UP else -1
            self.display.show(f"[{self.elevator_id}] Passing floor {self.current_floor}")

       self.requests.remove(target)
       self.display.show(f"[{self.elevator_id}] Arrived at floor {target.floor}, doors opening")

In [13]:
class SchedulingStrategy(ABC):
    @abstractmethod
    def next_stop(self, current_floor: int, requests: list, direction: 'Direction'):
        pass

In [14]:
class ElevatorController:
    def __init__(self, elevators:list['Elevator']):
      self.elevators = elevators

    def dispatch(self, request: 'Request'):
      best = min(self.elevators, key = lambda e : abs(e.current_floor - request.floor))
      best.add_request(request)
      return best
    def step_all(self):
      for elevator in self.elevators:
        elevator.step()

In [20]:
from enum import Enum
from abc import ABC, abstractmethod

class Direction(Enum):
    UP = 1
    DOWN = 2
    IDLE = 3

class Display(ABC):
    @abstractmethod
    def show(self, message:str):
      pass

class ConsoleDisplay(Display):
  def show(self, message:str):
     print(message)
class SilentDisplay(Display):
  def show(self, message:str):
    pass
elevator_a = Elevator("A", strategy=ScanStrategy(), display=ConsoleDisplay(), current_floor=0)
elevator_b = Elevator("B", strategy=NearestFirstStrategy(), display=ConsoleDisplay(), current_floor=5)

controller = ElevatorController([elevator_a, elevator_b])

In [22]:
class Request:
  def __init__(self, floor: int):
    self.floor = floor

  def __repr__(self):
    return f"Request(floor = { self.floor})"

controller.dispatch(Request(floor=4))
controller.dispatch(Request(floor=7))

controller.step_all()
controller.step_all()

[B] Request added for floor 4
[B] Request added for floor 7
[B] Passing floor 4
[B] Arrived at floor 4, doors opening
[B] Passing floor 5
[B] Passing floor 6
[B] Passing floor 7
[B] Arrived at floor 7, doors opening
